# **Prompting LLMs**

**Author:** Louis G. Binwag III

**Reference:** Course Notes (.ipynb)

---

**Instructions:** Evaluate the models on a subset of the BelebeleLinks to an external site. benchmark, specifically the Filipino subset. This task will require you to work with Filipino prompts.


1. **Load the dataset:** The Belebele dataset is available on HuggingFaceLinks to an external site.. You will need to load the tgl_Latn split (all 900 rows).

2. **Create Filipino prompts:** For each example in the dataset, you will need to construct a multiple-choice question in Filipino. The dataset contains a context, a question, and four possible answers.

3. **Evaluate the models:** Use the gemma3:1b, llama3.2:1b, and the quantized aisingapore/Gemma-SEA-LION-v3-9B-IT:q2_k models to answer the questions.

4. **Save the results:** Your output should be one JSONL results file for each model you test (e.g., belebele_results_gemma3:1b.jsonl). Each line in the file should be a JSON object containing the model_name, prompt, response, correct_answer, and whether the model's answer was correct.

**Expected Output:** Jupyter Notebook

In [28]:
!pip install --quiet datasets


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
from datasets import load_dataset

ds = load_dataset("facebook/belebele", "tgl_Latn")
split = 'validation' if 'validation' in ds else 'test' if 'test' in ds else 'train'

In [30]:
import pandas as pd
df = ds[split].to_pandas()
df.head()

,link,question_number,flores_passage,question,mc_answer1,mc_answer2,mc_answer3,mc_answer4,correct_answer_num,dialect,ds
0,https://en.wikibooks.org/wiki/Accordion/Right_...,1,Siguruhing kalmado ang iyong kamay hangga't po...,"Ayon sa sipi, ano ang hindi maituturing na tum...","Para sa mas malakas na volume, dagdagan ang pu...","Hanggang maaari, iwasan ang hindi kinakailanga...",Maging maingat sa pag-abot ng nota habang nana...,Mas bilisan pa ang paggalaw ng mga bellow upan...,1,tgl_Latn,2023-06-01
1,https://en.wikibooks.org/wiki/Accordion/Right_...,2,Siguruhing kalmado ang iyong kamay hangga't po...,"Kapag nagpapatugtog ng akordyon, alin sa mga s...",Mas mabilis na paggalaw,Mas malakas na puwersa,Mas mababang presyon,Mas kakaunting paggalaw ng daliri,1,tgl_Latn,2023-06-01
2,https://en.wikibooks.org/wiki/All_About_Conver...,1,Isa sa mga pinakakaraniwang problema kapag sin...,Bakit putol ang mga border sa mga imahe sa tel...,Upang makapaglagay ng subtitles,Upang mapuno ang buong screen ng imahe,Upang mapasimple ang pag-convert sa ibang format,Upang maging napakalapit ng subtitle sa ibaban...,2,tgl_Latn,2023-06-01
3,https://en.wikibooks.org/wiki/All_About_Conver...,2,Isa sa mga pinakakaraniwang problema kapag sin...,"Ayon sa sipi, alin sa mga sumusunod na problem...",Imahe na hindi mapupuno ang buong screen,Subtitles na bahagyang naputol,Image na mapupuno ang buong screen,Naputol na mga border,2,tgl_Latn,2023-06-01
4,https://en.wikibooks.org/wiki/American_Revolut...,2,Umasa ang plano ng Amerika sa paglulunsad ng m...,Saan may matatagpuan na Britanong garison?,Sapa ng Assunpink,Trenton,Bordentown,Princeton,3,tgl_Latn,2023-06-01


In [31]:
def build_prompt(item):
    """Builds a Filipino multiple-choice question prompt."""
    passage = item["flores_passage"]
    question = item["question"]
    a1 = item["mc_answer1"]
    a2 = item["mc_answer2"]
    a3 = item["mc_answer3"]
    a4 = item["mc_answer4"]

    prompt = (
        "Basahin ang sipi at sagutin ang tanong.\n\n"
        f"Flores Passage: {passage}\n\n"
        f"Question: {question}\n\n"
        "Choices:\n"
        f"(A) {a1}\n"
        f"(B) {a2}\n"
        f"(C) {a3}\n"
        f"(D) {a4}\n\n"
        "Ang tamang sagot ay:"
    )

    return prompt


In [32]:
def clean_response(text):
    """
    Extracts only the first A/B/C/D from Gemma’s response.
    """
    match = re.search(r"[ABCD]", text.upper())
    return f"({match.group(0)})" if match else None

In [37]:
!curl -fsSL https://ollama.com/install.sh | sh


'sh' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
import requests

def query_ollama(model, prompt):
    """Send a prompt to a locally running Ollama model and return its text response."""
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        },
        timeout=60
    )
    response.raise_for_status()
    return response.json()["response"].strip()


In [ ]:
# # !wsl --install
# # !curl -fsSL https://ollama.com/install.sh | sh

# import os
# import asyncio

# # Set LD_LIBRARY_PATH so the system can find ollama's shared libraries
# os.environ['LD_LIBRARY_PATH'] = '/usr/lib/x86_64-linux-gnu'

# async def run_ollama():
#     proc = await asyncio.create_subprocess_shell(
#         'ollama serve',
#         stdout=asyncio.subprocess.PIPE,
#         stderr=asyncio.subprocess.PIPE
#     )
#     # The server is now running in the background
#     print("Ollama server started.")
#     # We don't await proc.communicate() here to let it run in the background

# # Start the server
# await run_ollama()

# # Give the server a moment to start
# !sleep 5

NotImplementedError: 

In [4]:
!ollama pull gemma3:1b
!ollama pull llama3.2:1b
!ollama pull aisingapore/Gemma-SEA-LION-v3-9B-IT:q2_k


'ollama' is not recognized as an internal or external command,
operable program or batch file.
'ollama' is not recognized as an internal or external command,
operable program or batch file.
'ollama' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
!ollama list

'ollama' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import subprocess
import re


In [ ]:
def ask_gemma(prompt, model="gemma3:1b"):
    """
    Sends the prompt to the Ollama model and returns raw text output.
    """
    result = subprocess.run(
        ["ollama", "run", model],
        input=prompt.encode("utf-8"),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    return result.stdout.decode("utf-8").strip()

In [35]:
sample_item = df.iloc[0]
prompt = build_prompt(sample_item)
print("Prompt to Gemma:\n", prompt)

response = ask_gemma(prompt)
print("\n Answer (Gemma):\n", response)


Prompt to Gemma:
 Basahin ang sipi at sagutin ang tanong.

Flores Passage: Siguruhing kalmado ang iyong kamay hangga't posible habang wastong inaabot pa rin ang lahat ng mga nota - subukan ding huwag gumawa ng labis na paggalaw sa iyong mga daliri. Sa ganitong paraan, papagurin mo lamang ang sarili mo nang kaunti hangga't maaari. Tandaan, walang pangangailangan na pindutin nang madiin ang mga teklado para lumakas ang tunog katulad ng kapag sa piyano. Sa akordyon, upang makakuha ng karagdagang dami, gumamit ka ng mga bellow na mayroong higit na presyon at bilis.

Question: Ayon sa sipi, ano ang hindi maituturing na tumpak na paalala para sa matagumpay na pagpapatugtog ng akordyon?

Choices:
(A) Para sa mas malakas na volume, dagdagan ang puwersa ng pagpindot sa teklado
(B) Hanggang maaari, iwasan ang hindi kinakailangang paggalaw upang mapreserba ang iyong stamina
(C) Maging maingat sa pag-abot ng nota habang nananatiling kalmado ang kamay
(D) Mas bilisan pa ang paggalaw ng mga bellow u

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [34]:
from tqdm import tqdm
import json

num_to_letter = {1: "A", 2: "B", 3: "C", 4: "D"}
letter_to_num = {"A": 1, "B": 2, "C": 3, "D": 4}

model_name = "gemma3:1b"
output_file = f"belebele_results_{model_name.replace(':', '_')}.jsonl"

with open(output_file, "w") as f:
    for _, item in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating {model_name}"):
        prompt = build_prompt(item)
        response_raw = ask_gemma(prompt)
        response_clean = clean_response(response_raw).upper().strip()

        # --- map correct numeric label to letter ---
        correct_num = item["correct_answer_num"]
        correct_answer = f"({num_to_letter.get(correct_num, '?')})"

        # --- check if model response matches ---
        response_letter = response_clean.replace("(", "").replace(")", "")
        response_num = letter_to_num.get(response_letter, None)
        is_correct = (response_num == correct_num)

        result = {
            "model_name": model_name,
            "prompt": prompt,
            "response": f"({response_letter})",  # always wrapped like (A)
            "correct_answer": correct_answer,
            "correct": is_correct
        }

        f.write(json.dumps(result, ensure_ascii=False) + "\n")

print(f"Results saved to {output_file}")


Evaluating gemma3:1b:   0%|          | 0/900 [00:00<?, ?it/s]


FileNotFoundError: [WinError 2] The system cannot find the file specified